In [3]:
import requests
import pandas as pd

# A session makes sure our python script remembers things we did previously, like getting a cookie
# Without a session, we wouldn't retain the cookie we got from the front page and then we'd turn up to the API empty-handed
session = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",  # this is a standard web browser fingerprint
    "X-Requested-With": "XMLHttpRequest",  # makes it look like it's the website's JavaScript itself fetching the data to populate the page
}
# We don't need anything from the front page but we want to look like a normal human who first went to the
# front page, got a cookie, then our browser showed the cookie to the API and called it
print("Step 1: Establishing Session & Pulling Payload...")
session.get("https://understat.com/league/EPL", headers=headers)
# This is getting the actual data from the hidden football data API
response = session.get("https://understat.com/getLeagueData/EPL/2025", headers=headers)

if response.status_code == 200:  # if we get in successfully:
    raw_data = response.json()

    # This extractor will still work even if understats adds a new layer to their JSON data,
    # because it searches recursively until it finds what we want
    def find_teams(data_node):
        teams = []
        #
        if isinstance(data_node, dict):  # if data_node is a dictionary:
            if (
                "title" in data_node and "history" in data_node
            ):  # if data_node has keys "title" and "history":
                teams.append(data_node)
            else:
                for (
                    value
                ) in (
                    data_node.values()
                ):  # for every single value inside this dictionary (whether it's another dict, a list, a string, or a number):
                    teams.extend(
                        find_teams(value)
                    )  # recursively calls find_teams until it finds the data we want, then puts everything in there into the array we return
        elif isinstance(data_node, list):  # if data_node is a list:
            for item in data_node:  # for each item in this list:
                teams.extend(
                    find_teams(item)
                )  # recursively call find_teams until it finds the data we want, then puts everything in there into the array we return
        return teams

    print("Step 2: Executing Recursive Search to bypass the wrappers...")
    extracted_teams = find_teams(raw_data)

    league_data = []
    for team in extracted_teams:
        team_name = team["title"]
        matches_played = len(team["history"])
        # Sums up stats from every match in the "history" array.
        # Because the API URL specified 2025, it gives the 25/26 season total.
        total_xg = sum([match["xG"] for match in team["history"]])
        total_xga = sum([match["xGA"] for match in team["history"]])
        total_goals = sum([match["scored"] for match in team["history"]])

        league_data.append(
            {
                "Team": team_name,
                "Matches": matches_played,
                "Goals": total_goals,
                "xG": round(total_xg, 2),
                "xGA": round(total_xga, 2),
            }
        )

    df = (
        pd.DataFrame(league_data)
        .sort_values(by="xG", ascending=False)
        .reset_index(
            drop=True
        )  # gives the newly sorted table new row numbers, not keeping the old ones from before sorting,
        # and throws the old row numbers in the bin
    )

    print("\nData Extraction Complete. Displaying Top Attacking Teams:")
    display(df.head(10))

else:
    print("API Error.")

Step 1: Establishing Session & Pulling Payload...
Step 2: Executing Recursive Search to bypass the wrappers...

Data Extraction Complete. Displaying Top Attacking Teams:


,Team,Matches,Goals,xG,xGA
0,Chelsea,27,48,56.81,40.08
1,Arsenal,28,56,56.60,21.59
2,Manchester City,27,56,53.38,31.97
3,Manchester United,27,48,51.26,34.60
4,Liverpool,27,42,49.68,35.06
5,Brentford,27,40,48.19,39.60
6,Crystal Palace,27,29,46.34,40.11
7,Bournemouth,27,43,45.54,40.35
8,Newcastle United,27,38,44.32,35.54
9,Leeds,27,37,41.56,41.45


In [4]:
# Let's calculate the per-game averages first
df["xG_per_game"] = df["xG"] / df["Matches"]
df["xGA_per_game"] = df["xGA"] / df["Matches"]

# 2. Calculate the League Average xG per game
# Note: League Average xG and League Average xGA are mathematically identical over a full season
league_avg_xg_per_game = df["xG_per_game"].mean()

# 3. Calculate Relative Strengths
# Attack Strength > 1.0 means they create more xG than average.
df["Attack_Strength"] = df["xG_per_game"] / league_avg_xg_per_game

# Defense Strength < 1.0 means they concede less xGA than average (a good thing!).
df["Defense_Strength"] = df["xGA_per_game"] / league_avg_xg_per_game

print("\nModel Parameters Calculated:")
display(df[["Team", "Attack_Strength", "Defense_Strength"]].head(10))


Model Parameters Calculated:


,Team,Attack_Strength,Defense_Strength
0,Chelsea,1.394970,0.984165
1,Arsenal,1.340177,0.511209
2,Manchester City,1.310746,0.785024
3,Manchester United,1.258690,0.849603
4,Liverpool,1.219893,0.860899
5,Brentford,1.183306,0.972378
6,Crystal Palace,1.137879,0.984901
7,Bournemouth,1.118235,0.990795
8,Newcastle United,1.088278,0.872685
9,Leeds,1.020506,1.017805


In [ ]:
def calculate_match_lambdas(home_team, away_team, df, league_avg_xg):

    # IMPORTANT: These home_team and away_team variables DO NOT take into account home advantage. They are not xG
    # for that team at home, they are just naming conventions for the two teams so we can distinguish them

    # we will address this later to account for home advantage

    # 1. Extract the specific stats for the Home Team
    home_stats = df[df["Team"] == home_team].iloc[0]
    home_attack = home_stats["Attack_Strength"]
    home_defense = home_stats["Defense_Strength"]

    # 2. Extract the specific stats for the Away Team
    away_stats = df[df["Team"] == away_team].iloc[0]
    away_attack = away_stats["Attack_Strength"]
    away_defense = away_stats["Defense_Strength"]

    # 3. Calculate Lambda (Expected Goals for this specific match)
    # Home Expected Goals = Home Attack * Away Defense * League Average
    home_lambda = home_attack * away_defense * league_avg_xg

    # Away Expected Goals = Away Attack * Home Defense * League Average
    away_lambda = away_attack * home_defense * league_avg_xg

    return round(home_lambda, 3), round(away_lambda, 3)


# --- Test it out ---
# Make sure 'league_avg_xg_per_game' is defined from your previous cell
home_team_test = "Burnley"  # Replace with a valid name from your 'Team' column
away_team_test = "Liverpool"

home_xg, away_xg = calculate_match_lambdas(
    home_team_test, away_team_test, df, league_avg_xg_per_game
)

print(
    f"Projected Matchup: {home_team_test} ({home_xg} xG) vs {away_team_test} ({away_xg} xG)"
)

Projected Matchup: Arsenal (1.989 xG) vs Chelsea (1.076 xG)


In [8]:
import numpy as np
from scipy.stats import poisson
import pandas as pd


def calculate_match_probabilities(home_lambda, away_lambda, max_goals=5):
    # 1. Create an array of possible goals (0 through 5)
    goals_range = np.arange(0, max_goals + 1)  # creates an array from 0 to max_goals

    # 2. Calculate the independent Poisson probabilities for each team
    # poisson.pmf (Probability Mass Function) takes the goal range and the lambda
    home_probs = poisson.pmf(
        goals_range, home_lambda
    )  # creates an array of probs for each number in goals_range
    away_probs = poisson.pmf(goals_range, away_lambda)

    # 3. Build the Bivariate Matrix
    # np.outer multiplies every item in home_probs by every item in away_probs
    prob_matrix = np.outer(
        home_probs, away_probs
    )  # creates a 6x6 matrix for every combination of goals and says their probabilities

    # 4. Sum the matrix sections to get Match Odds
    home_win_prob = np.sum(
        np.tril(prob_matrix, k=-1)
    )  # tril means triangle-lower, setting k=-1 means we only keep everything below the main diagonal,
    # so where Home Goals > Away Goals

    # np.diag gets the main diagonal (Home Goals == Away Goals)
    draw_prob = np.sum(np.diag(prob_matrix))

    # np.triu(matrix, 1) gets everything above the diagonal (Home Goals < Away Goals)
    away_win_prob = np.sum(np.triu(prob_matrix, 1))

    # Let's wrap the matrix in a Pandas DataFrame so it looks nice if you print it
    matrix_df = pd.DataFrame(
        prob_matrix,
        columns=[f"Away_{i}" for i in range(max_goals + 1)],
        index=[f"Home_{i}" for i in range(max_goals + 1)],
    )

    return {
        "Home_Win_Prob": home_win_prob,
        "Draw_Prob": draw_prob,
        "Away_Win_Prob": away_win_prob,
        "Matrix": matrix_df,
    }


# --- Test the Engine ---
# We'll pass in the home_xg and away_xg you calculated in the previous step
match_results = calculate_match_probabilities(home_xg, away_xg, max_goals=5)

print("--- Model Predictions ---")
print(f"Home Win:  {match_results['Home_Win_Prob'] * 100:.2f}%")
print(f"Draw:      {match_results['Draw_Prob'] * 100:.2f}%")
print(f"Away Win:  {match_results['Away_Win_Prob'] * 100:.2f}%\n")

print("--- Exact Scoreline Probability Matrix ---")
# Multiply by 100 and round to 2 decimals for readability
display(match_results["Matrix"].map(lambda x: f"{x*100:.2f}%"))

--- Model Predictions ---
Home Win:  56.92%
Draw:      21.45%
Away Win:  19.93%

--- Exact Scoreline Probability Matrix ---


,Away_0,Away_1,Away_2,Away_3,Away_4,Away_5
Home_0,4.67%,5.02%,2.70%,0.97%,0.26%,0.06%
Home_1,9.28%,9.98%,5.37%,1.93%,0.52%,0.11%
Home_2,9.23%,9.93%,5.34%,1.92%,0.52%,0.11%
Home_3,6.12%,6.58%,3.54%,1.27%,0.34%,0.07%
Home_4,3.04%,3.27%,1.76%,0.63%,0.17%,0.04%
Home_5,1.21%,1.30%,0.70%,0.25%,0.07%,0.01%


In [9]:
import pandas as pd


def find_value_bets(match_results, bookmaker_odds):
    bets_analysis = []

    # Map the bookmaker keys to your model's probability keys
    outcomes = {"Home": "Home_Win_Prob", "Draw": "Draw_Prob", "Away": "Away_Win_Prob"}

    for outcome, prob_key in outcomes.items():
        # Get your model's true probability
        model_prob = match_results[prob_key]

        # Convert to implied decimal odds (handle division by zero just in case)
        model_odds = 1 / model_prob if model_prob > 0 else float("inf")

        # Get the bookmaker's odds
        bookie_odds = bookmaker_odds.get(outcome, 0)

        # Calculate Expected Value (EV)
        # EV = (True Probability * Bookmaker Decimal Odds) - 1
        ev = (model_prob * bookie_odds) - 1

        bets_analysis.append(
            {
                "Market": outcome,
                "Model_Prob": f"{model_prob * 100:.2f}%",
                "Model_Odds": round(model_odds, 2),
                "Bookie_Odds": bookie_odds,
                "Edge (EV)": f"{ev * 100:.2f}%",
                "Bet_Signal": "🔥 VALUE (BET)" if ev > 0 else "PASS",
            }
        )

    return pd.DataFrame(bets_analysis)


# --- Test the Arbitrage Scanner ---
# Let's pretend Pinnacle has these odds for the match:
# Home: 2.10, Draw: 3.50, Away: 3.20
pinnacle_odds = {"Home": 2.10, "Draw": 3.50, "Away": 3.20}

# Pass in the 'match_results' dictionary we generated in Step 3
value_dashboard = find_value_bets(match_results, pinnacle_odds)

print("--- Market Analysis Dashboard ---")
display(value_dashboard)

--- Market Analysis Dashboard ---


,Market,Model_Prob,Model_Odds,Bookie_Odds,Edge (EV),Bet_Signal
0,Home,56.92%,1.76,2.1,19.54%,🔥 VALUE (BET)
1,Draw,21.45%,4.66,3.5,-24.94%,PASS
2,Away,19.93%,5.02,3.2,-36.23%,PASS
